Resilient datasets 

In [1]:
from pyspark.sql import SparkSession

# Create RDDs from different sources

spark = SparkSession.builder.appName("population-analysis").getOrCreate()

population_rdd = spark.sparkContext.textFile("world_population.csv")

numbers_rdd = spark.sparkContext.parallelize([10, 20, 30, 40, 50])

sample_lines = population_rdd.take(3)

sample_numbers = numbers_rdd.take(3)

print(sample_lines)
print(sample_numbers)


['location,population,percent_of_world,date_updated', 'India,1413324000,17.3%,1 Mar 2025', 'China,1408280000,17.2%,31 Dec 2024']
[10, 20, 30]


Removing Headers - and using .first() and .filter() transformation

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("population-analysis").getOrCreate()
population_rdd = spark.sparkContext.textFile("world_population.csv")

# Explore and clean the population data

total_rows = population_rdd.count()
header = population_rdd.first()
clean_data = population_rdd.filter(lambda line: line != header)
data_rows = clean_data.count()
preview_data = clean_data.take(5)
print(header)
print(total_rows)
print(data_rows)
print(preview_data)

location,population,percent_of_world,date_updated
240
239
['India,1413324000,17.3%,1 Mar 2025', 'China,1408280000,17.2%,31 Dec 2024', 'United States,340110988,4.2%,1 Jul 2024', 'Indonesia,282477584,3.5%,30 Jun 2024', 'Pakistan,241499431,3.0%,1 Mar 2023']


Transforming data - cleaning the data and using .map() transformation to prepare for analysis

In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("population-analysis").getOrCreate()
population_rdd = spark.sparkContext.textFile("world_population.csv")
header = population_rdd.first()
clean_data = population_rdd.filter(lambda line: line != header)

# Transform raw strings into structured data

split_data = clean_data.map(lambda line: line.split(','))
country_population = split_data.map(lambda fields: (fields[0], int(fields[1])))
country_count = country_population.count()
sample_structured = country_population.take(5)
print(country_count, sample_structured, sep="\n")

239
[('India', 1413324000), ('China', 1408280000), ('United States', 340110988), ('Indonesia', 282477584), ('Pakistan', 241499431)]


Fitlering and data quality :-

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("population-analysis").getOrCreate()
population_rdd = spark.sparkContext.textFile("world_population.csv")
header = population_rdd.first()
clean_data = population_rdd.filter(lambda line: line != header)
split_data = clean_data.map(lambda line: line.split(','))
country_population = split_data.map(lambda fields: (fields[0], int(fields[1])))

# Filter countries by population size


large_countries = country_population.filter(lambda pair: pair[1] > 50_000_000)
small_countries = country_population.filter(lambda pair: pair[1] < 1_000_000)
large_count = large_countries.count()
small_list = small_countries.collect()
print(large_count, small_list[:10], sep="\n")

30 
 [('Cyprus', 966400), ('Fiji', 900869), ('Comoros', 870038), ('Bhutan', 777224), ('Guyana', 772975), ('Solomon Islands', 750325), ('Macau (China)', 687900), ('Luxembourg', 681973), ('Montenegro', 623327), ('Suriname', 616500)]


DAGs and Lazy evaluations - Building transformation chains

In [6]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("population-analysis").getOrCreate()
population_rdd = spark.sparkContext.textFile("world_population.csv")
header = population_rdd.first()
clean_data = population_rdd.filter(lambda line: line != header)
split_data = clean_data.map(lambda line: line.split(','))
country_population = split_data.map(lambda fields: (fields[0], int(fields[1])))

# Build a complex transformation pipeline
large_population_pipeline=(country_population
                           .filter(lambda pair : pair[1] > 100_000_000)
                           .map(lambda pair: (pair[0],pair[1]/1_000_000))
                           .filter(lambda pair : pair[1] > 200)
                           )
final_count = large_population_pipeline.count()
final_results = large_population_pipeline.collect()
print(final_count, final_results, sep = '\n')

7
[('India', 1413.324), ('China', 1408.28), ('United States', 340.110988), ('Indonesia', 282.477584), ('Pakistan', 241.499431), ('Nigeria', 223.8), ('Brazil', 212.58375)]


Aggregating the data : using .reduce() transformation 

In [7]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("population-analysis").getOrCreate()
population_rdd = spark.sparkContext.textFile("world_population.csv")
header = population_rdd.first()
clean_data = population_rdd.filter(lambda line: line != header)
split_data = clean_data.map(lambda line: line.split(','))
country_population = split_data.map(lambda fields: (fields[0], int(fields[1])))

# Calculate global population statistics
smallest_country = country_population.reduce(lambda x,y : x if x[1] < y[1] else y)
largest_country = country_population.reduce(lambda x,y : x if x[1] > y[1] else y)

populations = country_population.map(lambda pair : pair[1])

total_population = populations.reduce(lambda x,y : x+y)

avg_population = total_population/populations.count()

print(total_population,avg_population, sep='\n')

7982296746
33398731.154811714


Key-value Operations - Using .reduceByKey() transformation

In [8]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("population-analysis").getOrCreate()
population_rdd = spark.sparkContext.textFile("world_population.csv")
header = population_rdd.first()
clean_data = population_rdd.filter(lambda line: line != header)
split_data = clean_data.map(lambda line: line.split(','))
country_population = split_data.map(lambda fields: (fields[0], int(fields[1])))

# Analyze population distribution by categories

def categorize_population(population):
    if population >= 100_000_000:
        return "Very Large"
    elif population >= 10_000_000:
        return "Large"
    elif population >= 1_000_000:
        return "Medium"
    else:
        return "Small"
    
population_categories = country_population.map(lambda pair: (categorize_population(pair[1]),1))

category_counts = population_categories.reduceByKey(lambda x,y:x+y)
results = category_counts.collect()
print(results)

[('Medium', 65), ('Small', 79), ('Very Large', 16), ('Large', 79)]
